## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")
groq_api_key



In [3]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x12b2a87c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12b2a8ac0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer")])

AIMessage(content='Hello Krish! 👋 It’s great to meet the Chief AI Engineer. How can I support you today—whether it’s brainstorming ideas, diving into technical details, or just chatting about the latest in AI? Let me know what’s on your mind!', additional_kwargs={'reasoning_content': 'The user says "Hi , My name is Krish and I am a Chief AI Engineer". They are introducing themselves. The assistant should respond appropriately. We need to maintain a friendly, professional tone. We can ask about their work or how we can help. Maybe ask what they need. The assistant is ready to help. We should greet them.'}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 85, 'total_tokens': 217, 'completion_time': 0.136931267, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.004159011, 'prompt_tokens_details': None, 'queue_time': 0.210857445, 'total_time': 0.141090278}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service

In [5]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer"),
        AIMessage(content="Hello Krish! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='You’re Krish, and you’re a **Chief AI Engineer**. In that role you’re likely responsible for overseeing AI strategy, architecture, and deployment across your organization—leading teams, setting technical standards, evaluating new models and tools, and ensuring that AI solutions align with business goals. Let me know if there’s anything specific you’d like to dive into!', additional_kwargs={'reasoning_content': 'The user says: "Hey What\'s my name and what do I do?" The user previously said "Hi , My name is Krish and I am a Chief AI Engineer". So we should answer: Your name is Krish and you are a Chief AI Engineer. Also we can elaborate. The user might want confirmation or elaboration. So answer accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 152, 'total_tokens': 306, 'completion_time': 0.163329288, 'completion_tokens_details': {'reasoning_tokens': 72}, 'prompt_time': 0.00775963, 'prompt_tokens_details': None, 'queue_time'

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [ ]:
!pip install langchain_community

In [92]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

/Users/amitmungare/Amit/myProject/agentic_ai/langchain/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [93]:
config={"configurable":{"session_id":"chat1"}}

In [94]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Amit and I am a Chief AI Engineer")],
    config=config
)

In [95]:
response.content

'Hello Amit! 👋  \nIt’s great to meet a Chief AI Engineer. How can I assist you today? Whether you’re looking for insights on the latest AI trends, help with a specific project, or just want to bounce ideas around, I’m here to help.'

In [96]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Amit.', additional_kwargs={'reasoning_content': 'User says "What\'s my name?" They previously said "Hi , My name is Amit and I am a Chief AI Engineer". So answer "Your name is Amit."'}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 153, 'total_tokens': 201, 'completion_time': 0.049401206, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.008569813, 'prompt_tokens_details': None, 'queue_time': 0.319732712, 'total_time': 0.057971019}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1078c-67e6-70e3-b1a8-551f6daa3b5a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 48, 'total_tokens': 201, 'output_token_details': {'reasoning': 34}})

In [97]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'I’m sorry, but I can’t help with that.'

In [98]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

'Nice to meet you, John! How can I help you today?'

In [99]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [100]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [101]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Amit")]})

AIMessage(content='Hello Amit! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hi My name is Amit". We need to respond as a helpful assistant. Likely greet and ask how can I help.'}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 95, 'total_tokens': 146, 'completion_time': 0.05612285, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.005301568, 'prompt_tokens_details': None, 'queue_time': 0.307136687, 'total_time': 0.061424418}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e594c51153', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1078c-6fe1-77c3-866d-0ac8b3c3538e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 95, 'output_tokens': 51, 'total_tokens': 146, 'output_token_details': {'reasoning': 30}})

In [102]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

/Users/amitmungare/Amit/myProject/agentic_ai/langchain/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [103]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Amit")],
    config=config
)

response

AIMessage(content='Hello Amit! 👋 How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond politely, greet, ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 95, 'total_tokens': 130, 'completion_time': 0.036276958, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.004616154, 'prompt_tokens_details': None, 'queue_time': 0.33373856, 'total_time': 0.040893112}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_66f3850a1a', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1078c-71aa-76d2-9e36-48b8436cf6d9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 95, 'output_tokens': 35, 'total_tokens': 130, 'output_token_details': {'reasoning': 14}})

In [104]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Amit.'

In [105]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [106]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Amit")],"language":"marathi"})
response.content

'नमस्कार अमित! तुम्हाला कसे वाटते? तुम्हाला काही विचारायचे किंवा चर्चा करायची आहे का? मी तुमची मदत करण्यास तयार आहे.'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [107]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

/Users/amitmungare/Amit/myProject/agentic_ai/langchain/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [108]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Amit")],"language":"Marathi"},
    config=config
)
repsonse.content

'नमस्कार अमित! आपण कसे आहात?'

In [109]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Marathi"},
    config=config,
)

In [110]:
response.content

'तुमचे नाव अमित आहे.'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [111]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [112]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'I don’t know for sure—only you can tell me what flavors you’re craving!  \nIf you’re looking for suggestions, some people love classic flavors like **chocolate, vanilla, or strawberry**, while others enjoy more adventurous options like **salted caramel, mint chocolate chip, or cookie‑dough**.  \nWhat’s your go‑to flavor? Or do you have a favorite brand? I’d love to hear!'

In [113]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked for the sum of 2 and 2: “What’s 2\u202f+\u202f2?”'

In [114]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

/Users/amitmungare/Amit/myProject/agentic_ai/langchain/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [115]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'I’m not sure what your name is—could you tell me?'

In [116]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'I don’t see any math problem in our conversation so far. If you have a specific question or problem you’d like help with, just let me know and I’ll be happy to assist!'